# BERT Fine-Tuning — Customer Review Classification

Split from the original combined demo (Task 2). Fine-tunes **bert-base-uncased** for
5-class review rating classification (HuggingFace Transformers).

**Compute:** run on **Serverless GPU (AI Runtime)** — select an **A10** accelerator in the
Environment panel before running.

**Source table:** `shm_skunkworks_catalog.cuj.reviews`

> Note: cells reordered vs the original so the notebook runs top-to-bottom — the model/
> dataset are defined (training cell) *before* the batch-inspection cells that use them.

In [ ]:
%pip install --quiet "transformers>=4.44" datasets accelerate torch scikit-learn mlflow
dbutils.library.restartPython()

## Setup — self-contained data generation
Creates the reviews table if it does not exist. Safe to re-run.

In [ ]:
# --- Setup: generate the source data (self-contained; idempotent) ---
spark.sql("CREATE SCHEMA IF NOT EXISTS shm_skunkworks_catalog.cuj")
spark.sql("""
CREATE TABLE IF NOT EXISTS shm_skunkworks_catalog.cuj.reviews AS
SELECT element_at(
    map(1, array('Terrible, broke on day one.','Awful experience, would not recommend.','Complete waste of money, very disappointed.'),
        2, array('Below average, several issues.','Not great, expected more for the price.','Meh, would probably not buy again.'),
        3, array('It was okay, nothing special.','Average product, does the job.','Fine but unremarkable overall.'),
        4, array('Pretty good, would buy again.','Solid quality, happy with it.','Works well, minor nitpicks only.'),
        5, array('Absolutely love it, exceeded expectations!','Fantastic, best purchase this year.','Outstanding quality, highly recommend!'))[Rating],
    cast(floor(rand()*3)+1 AS int)) AS Review, Rating
FROM (SELECT explode(sequence(1,600)) AS id, cast(floor(rand()*5)+1 AS int) AS Rating)
""")
print("reviews ready:", spark.table("shm_skunkworks_catalog.cuj.reviews").count(), "rows")

In [ ]:
reviews_df = spark.read.table("shm_skunkworks_catalog.cuj.reviews")
display(reviews_df)

In [ ]:
from sklearn.model_selection import train_test_split

reviews_pd_df = reviews_df.toPandas()
reviews_pd_df.rename(columns={"Rating": "labels"}, inplace=True)
train_df, val_df = train_test_split(reviews_pd_df, test_size=0.2, random_state=42)
train_df["labels"] = train_df["labels"] - 1
val_df["labels"] = val_df["labels"] - 1

display(train_df)

In [ ]:
print("Train label values:", sorted(train_df["labels"].unique()))
print("Val label values:", sorted(val_df["labels"].unique()))
print("Train label dtype:", train_df["labels"].dtype)


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch

# Disable WandB
os.environ["WANDB_DISABLED"] = "true"

# 3. Convert to Hugging Face dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

# 4. Tokenizer
checkpoint = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example["Review"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

# 5. Model
model = BertForSequenceClassification.from_pretrained(checkpoint, num_labels=5)

# 6. Define Trainer
training_args = TrainingArguments(
    output_dir="/tmp/results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="/tmp/logs",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# 7. Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 8. Train
trainer.train()

In [ ]:
from torch.utils.data import DataLoader

dl = DataLoader(train_ds.with_format("torch"), batch_size=200)
batch = next(iter(dl))

print("Batch labels:", batch["labels"])
print("Max label:", batch["labels"].max().item())
print("Min label:", batch["labels"].min().item())

In [ ]:
model.eval()
with torch.no_grad():
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )
print("Loss:", out.loss)
print("Logits:", out.logits)
